Пакет SURPRISE:

1) используйте данные MovieLens 1M,
2) можно использовать любые модели из пакета,
3) получите RMSE на тестовом сете 0,87 и ниже.
Комментарий преподавателя:
В домашнем задании на датасет 1М может не хватить RAM. Можно сделать на 100K. Качество RMSE предлагаю считать на основе Cross-validation (5 фолдов), а не на отложенном датасете.

In [1]:
!pip install surprise

  Using cached scikit_surprise-1.1.4-cp312-cp312-macosx_11_0_arm64.whl


In [2]:
!pip install numpy==1.26.4

In [75]:
from surprise import KNNWithMeans
from surprise import Dataset
from surprise import accuracy
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise.model_selection import GridSearchCV

import pandas as pd
import numpy as np

In [4]:
data_movies = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/RS/collaborative RS/ml-latest-small/movies.csv')
data_ratings = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/RS/collaborative RS/ml-latest-small/ratings.csv')

In [5]:
data_movies_with_ratings = data_movies.join(data_ratings.set_index('movieId'), on='movieId').reset_index(drop=True)

In [6]:
data_movies_with_ratings

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,4.0,9.649827e+08
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0,8.474350e+08
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7.0,4.5,1.106636e+09
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15.0,2.5,1.510578e+09
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17.0,4.5,1.305696e+09
...,...,...,...,...,...,...
100849,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,184.0,4.0,1.537109e+09
100850,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,184.0,3.5,1.537110e+09
100851,193585,Flint (2017),Drama,184.0,3.5,1.537110e+09
100852,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,184.0,3.5,1.537110e+09


In [7]:
dataset = pd.DataFrame({
    'uid': data_movies_with_ratings.userId,
    'iid': data_movies_with_ratings.title,
    'rating':data_movies_with_ratings.rating
})

In [9]:
data_ratings.rating.min()

0.5

In [10]:
data_ratings.rating.max()

5.0

In [28]:
data_ratings.rating.unique()

array([4. , 5. , 3. , 2. , 1. , 4.5, 3.5, 2.5, 0.5, 1.5])

In [29]:
data_ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [11]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(dataset, reader)

In [46]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=1)

In [54]:
train_users = set(trainset.to_raw_uid(u) for u in trainset.all_users()) ## получаем кортеж исходных id пользователя с помощью функции to_raw_uid, которая возвращает оригинальные данные
train_items = set(trainset.to_raw_iid(i) for i in trainset.all_items())

# Оставляем только те записи из testset, у которых юзер и фильм есть в trainset
filtered_testset = [(u, i, r) for (u, i, r) in testset 
                    if (u in train_users) and (i in train_items)]

In [77]:
param_grid = {
    'k': [10, 20, 30],
    'sim_options_name': ['cosine', 'pearson'],
    'sim_options_user_based': [True, False]
}

gs = GridSearchCV(KNNWithMeans, param_grid, measures=['rmse'], cv=5,joblib_verbose=2)
gs.fit(data)


Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   26.4s


Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [78]:
print("Лучший RMSE:", gs.best_score['rmse'])
print("Лучшие параметры:", gs.best_params['rmse'])

Лучший RMSE: nan
Лучшие параметры: {'k': 10, 'sim_options_name': 'cosine', 'sim_options_user_based': True}


Проверии эффективность модели user-based

In [79]:
algo = KNNWithMeans(k=10, sim_options={
    'name': 'cosine',
    'user_based': True  # compute  similarities between users
})
algo.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [80]:
from surprise.model_selection import cross_validate
cross_validate(algo, data, measures=['RMSE','MAE'], cv=3, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNWithMeans on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    nan     nan     nan     nan     nan     
MAE (testset)     nan     nan     nan     nan     nan     
Fit time          0.05    0.06    0.06    0.06    0.00    
Test time         0.75    0.91    0.74    0.80    0.08    


{'test_rmse': array([nan, nan, nan]),
 'test_mae': array([nan, nan, nan]),
 'fit_time': (0.0549771785736084, 0.06096482276916504, 0.060324668884277344),
 'test_time': (0.7520749568939209, 0.9080319404602051, 0.7407753467559814)}

Кросс-валидация показывает значения метрик Nan, потому что не изменен холодный старт, то есть в тестовую выборку попадают объекты, которых нет в обучающей выборке. При этом, если отфильтроватьть тестовую выборку, и оставить в ней только те объекты, которые есть в обучающей, то модель с подобранными параметрами покажет RMSE в 0,89. Данное число не соответсвует требуемому, поэтому попробуем построить модель SVD, которая не зависит от холодного старта

In [73]:
test_pred = algo.test(filtered_testset)
accuracy.rmse(test_pred, verbose=True)

RMSE: 0.8977


0.8976618541426062

Модель SVD

In [81]:
param_grid_SVD = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30, 50],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.05, 0.1]
}

gs = GridSearchCV(SVD, param_grid_SVD, measures=['rmse'], cv=5,joblib_verbose=2)
gs.fit(data)


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   22.9s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:  2.3min
[Parallel(n_jobs=1)]: Done 364 tasks      | elapsed:  6.6min


In [82]:
print("Лучший RMSE:", gs.best_score['rmse'])
print("Лучшие параметры:", gs.best_params['rmse'])

Лучший RMSE: nan
Лучшие параметры: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.002, 'reg_all': 0.02}


In [83]:
algo = SVD(n_factors=50,n_epochs=20,lr_all=0.002,reg_all=0.02, )
algo.fit(trainset)

In [84]:
test_pred = algo.test(filtered_testset)
accuracy.rmse(test_pred, verbose=True)

RMSE: 1.7968


1.7968289564392643

Получили очень большую метрику RMSE, значит, в данном случае модель не подходит

Попробуем самостоятельно разбить выборку на обучающую и тестовую, чтобы исключить влияние холодного старта на метрики


In [98]:
def train_test_no_cold_start(df, test_size=0.2, random_state=1):
    train_rows = []
    test_rows = []

    grouped = df.groupby('uid')
    for uid, group in grouped:
        n_test = max(1, int(len(group) * test_size)) ## создаем размер тестовой выборки
        group = group.sample(frac=1, random_state=random_state) ## перемешиваем группу
        test_rows.append(group.iloc[:n_test]) ## добавляем размер тестовой выборки в тест
        train_rows.append(group.iloc[n_test:]) ## остальное добавляем в train
    
     ## объединяем все строки для каждого отедльного пользователя в один дата-фрейм
    train_df = pd.concat(train_rows)
    test_df = pd.concat(test_rows)

#
    train_users = set(train_df['uid']) ## получаем пользователей из обучающей выборки
    train_items = set(train_df['iid']) ## фильмы
    test_df = test_df[test_df['uid'].isin(train_users) & test_df['iid'].isin(train_items)] ## оставляем только те объекты, которые есть в обучающей выборке

    return train_df, test_df

In [99]:
train_df,test_df = train_test_no_cold_start(dataset)

In [100]:
train_df

,uid,iid,rating
21929,1.0,"Three Caballeros, The (1945)",5.0
42931,1.0,Very Bad Things (1998),5.0
14106,1.0,Schindler's List (1993),5.0
7860,1.0,Pulp Fiction (1994),3.0
38227,1.0,Back to the Future Part III (1990),4.0
...,...,...,...
84996,610.0,28 Weeks Later (2007),4.0
91122,610.0,Iron Man 2 (2010),3.5
95796,610.0,Pacific Rim (2013),4.5
57990,610.0,Nurse Betty (2000),3.5


In [104]:
reader = Reader(rating_scale=(0.5, 5.0))
trainset_new = Dataset.load_from_df(train_df[['uid','iid','rating']], reader).build_full_trainset() ## превращает Dataset в объект Trainset, который понимает KNN, SVD и другие алгоритмы
testset_new = list(zip(test_df['uid'], test_df['iid'], test_df['rating']))

Так как GridSEarch самостоятельно разбивает данные на тест и обучение, то найдем лушчие парамеры самостоятельно через цикл

In [122]:
param_grid = {
    'k': [10, 20, 30],
    'sim_options_name': ['cosine', 'pearson'],
    'sim_options_user_based': [True, False]
}

In [125]:

def GridSearchByMe(param,train,test):
    best_rmse = float('inf')
    best_params = {}
    for k in param['k']:
        for name in param['sim_options_name']:
            for user_based in param['sim_options_user_based']:
                algo = KNNWithMeans(k=k, sim_options={'name': name, 'user_based': user_based})
            
                algo.fit(train)

                predictions = algo.test(test)
                
                rmse = accuracy.rmse(predictions, verbose=False)
                
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_params = {'k': k, 'name': name, 'user_based': user_based}
    print("Лучший RMSE:", best_rmse)
    print("Лучшие параметры:", best_params)


In [126]:
GridSearchByMe(param_grid,trainset_new,testset_new)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Лучший RMSE: 0.8417190239016518
Лучшие параметры: {'k': 30, 'name': 'pearson', 'us

In [110]:
algo_new = KNNWithMeans(k=30, sim_options={
    'name': 'pearson',
    'user_based': True  
})
algo_new.fit(trainset_new)

Computing the pearson similarity matrix...
Done computing similarity matrix.


In [111]:
test_pred = algo_new.test(testset_new)
accuracy.rmse(test_pred, verbose=True)

RMSE: 0.8921


0.8921426403029818

Все равно значение RMSE выше нормы, попробуем удалить фильмы, в которых стоит меньше 20 оценок

In [113]:
title_counts = data_movies_with_ratings['title'].value_counts()

titles_20 = title_counts[title_counts >= 20].index

data_movies_with_ratings_20 = data_movies_with_ratings[data_movies_with_ratings['title'].isin(titles_20)]

In [114]:
data_movies_with_ratings_20

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,4.0,9.649827e+08
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0,8.474350e+08
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7.0,4.5,1.106636e+09
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15.0,2.5,1.510578e+09
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17.0,4.5,1.305696e+09
...,...,...,...,...,...,...
100348,168252,Logan (2017),Action|Sci-Fi,567.0,4.0,1.525284e+09
100349,168252,Logan (2017),Action|Sci-Fi,586.0,5.0,1.529899e+09
100350,168252,Logan (2017),Action|Sci-Fi,596.0,5.0,1.535627e+09
100351,168252,Logan (2017),Action|Sci-Fi,599.0,3.5,1.498530e+09


In [118]:
dataset_20 = pd.DataFrame({
    'uid': data_movies_with_ratings_20.userId,
    'iid': data_movies_with_ratings_20.title,
    'rating':data_movies_with_ratings_20.rating
})

In [116]:
print(data_movies_with_ratings_20.rating.min(),data_movies_with_ratings_20.rating.max())

0.5 5.0


In [119]:
train_df_20,test_df_20 = train_test_no_cold_start(dataset_20)

In [140]:
train_df_20

,uid,iid,rating
50266,1.0,Who Framed Roger Rabbit? (1988),5.0
22443,1.0,That Thing You Do! (1996),4.0
32842,1.0,Face/Off (1997),5.0
41577,1.0,Edward Scissorhands (1990),5.0
49249,1.0,Total Recall (1990),4.0
...,...,...,...
50122,610.0,Time Bandits (1981),5.0
54218,610.0,Do the Right Thing (1989),5.0
26421,610.0,Army of Darkness (1993),4.5
66107,610.0,And Your Mother Too (Y tu mamá también) (2001),4.5


In [128]:

trainset_new_20 = Dataset.load_from_df(train_df_20[['uid','iid','rating']], reader).build_full_trainset() ## превращает Dataset в объект Trainset, который понимает KNN, SVD и другие алгоритмы
testset_new_20= list(zip(test_df_20['uid'], test_df_20['iid'], test_df_20['rating']))

In [129]:
GridSearchByMe(param_grid,trainset_new_20,testset_new_20)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Лучший RMSE: 0.8417190239016518
Лучшие параметры: {'k': 30, 'name': 'pearson', 'us

Удалось достичь метрики RMSE ниже пороговой! Обучим модель на этих параметрах

In [130]:
algo_new.fit(trainset_new_20)

Computing the pearson similarity matrix...
Done computing similarity matrix.


In [131]:
test_pred = algo_new.test(testset_new_20)
accuracy.rmse(test_pred, verbose=True)

RMSE: 0.8417


0.8417190239016518

In [141]:
new_pred = algo_new.predict(uid=2, iid='Who Framed Roger Rabbit? (1988)')
new_pred

Prediction(uid=2, iid='Who Framed Roger Rabbit? (1988)', r_ui=None, est=3.828378991506089, details={'actual_k': 19, 'was_impossible': False})

Вывод: в ходе работы была обучена коллаборативная модель предсказаний, основанная на взаимодействии пользователей. Было определено экспериментально, что очень важно для качественной работы модели, чтобы
а) был исключен холодный старт, то есть чтобы тестовая выборка встречалась уже в обучающей
б) не было слишком редких фильмов, которых оценили маленькое количество пользователей (в данном примере выбрано пороговое значение в 20 пользователей)
В связи с этими выводами пришлось самостоятельно разбивать данные на тестовую и обучающую выборки и самостоятельно производить поиск лучших параметров модели KNNWithMeans. В ходе этого было выяснено, что модель с данными параметрами {'k': 30, 'name': 'pearson', 'user_based': True} показывает RMSE = 0,84, что ниже установленного в задании значения. 

--- Функция с урока ---

In [132]:
def generate_recommendation(uid, model, dataset, thresh=4, amount=5):
    all_titles = list(dataset['iid'].values)
    users_seen_titles = dataset[dataset['uid'] == uid]['iid']
    titles = np.array(list(set(all_titles) - set(users_seen_titles)))

    np.random.shuffle(titles)
    
    rec_list = []
    predictions = []
    for title in titles:
        review_prediction = model.predict(uid=uid, iid=title)
        predictions.append(review_prediction)  # добавляем в список
        
        rating = review_prediction.est
        

        if rating >= thresh:
            rec_list.append((title, round(rating, 2)))
            
            if len(rec_list) >= amount:
                return rec_list


In [136]:
generate_recommendation(2, algo, dataset)

[('Seven Samurai (Shichinin no samurai) (1954)', 4.19),
 ('Pulp Fiction (1994)', 4.26),
 ('Streetcar Named Desire, A (1951)', 4.04),
 ('District 9 (2009)', 4.02),
 ('Star Wars: Episode V - The Empire Strikes Back (1980)', 4.12)]